# XGBoost ile 30 gün sonrası baraj doluluk tahmini (Sinem)
Veri ve bölme ortak dosyalardan geliyor (`src/veri.py`, `src/degerlendir.py`), böylece LSTM ile aynı şekilde ölçülüyor.
Eğitim 2011-2022, test 2023. Model ayarı: 2011-2020 ile eğit, 2021-2022'de ölç.

In [ ]:
import sys; sys.path.append("..")
import itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from src.veri import birlesik_veri
from src.degerlendir import bol, metrikler, UFUK

df = birlesik_veri("../data/raw/")
print(df.shape, df.index.min().date(), df.index.max().date())

## Öznitelikler
Sadece o güne kadar bilinen bilgiler: bugünkü doluluk, son 7/30/90 günlük doluluk değişimi, yağış toplamı, ortalama tüketim, mevsim.
Hedef: 30 gün sonraki doluluk - bugünkü doluluk (değişim). Tahmin bugünkü doluluğa eklenerek 30 gün sonraki doluluk bulunur.

In [ ]:
def oznitelikler(df):
    X = pd.DataFrame(index=df.index)
    X["doluluk"] = df["toplam"]
    for g in [7, 30, 90]:
        X[f"degisim_{g}"] = df["toplam"] - df["toplam"].shift(g)
        X[f"yagis_{g}"] = df["yagis"].rolling(g).sum()
        X[f"tuketim_{g}"] = df["tuketim"].rolling(g).mean() / 1e6
    X["mevsim_sin"] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    X["mevsim_cos"] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
    return X

X = oznitelikler(df)
oz = list(X.columns)
veri = X.assign(hedef=df["toplam"].shift(-UFUK) - df["toplam"]).dropna()
p = bol(veri)
print({k: len(v) for k, v in p.items()})

def olc(parca, tahmin_degisim):
    return metrikler(parca["doluluk"] + parca["hedef"], parca["doluluk"] + tahmin_degisim)

## Model ayarı (doğrulama kümesi 2021-2022)
Küçük bir ızgara araması: ağaç derinliği, öğrenme hızı, ağaç sayısı. En düşük doğrulama MAE'sini veren ayar seçilir.

In [ ]:
izgara = {"max_depth": [2, 3, 4, 6], "learning_rate": [0.03, 0.1], "n_estimators": [200, 500],
          "min_child_weight": [1, 10]}
denemeler = []
for degerler in itertools.product(*izgara.values()):
    ayar = dict(zip(izgara, degerler))
    m = XGBRegressor(**ayar, subsample=0.8, colsample_bytree=0.8, random_state=0)
    m.fit(p["egitim"][oz], p["egitim"]["hedef"])
    denemeler.append({**ayar, **olc(p["dogrulama"], m.predict(p["dogrulama"][oz]))})
denemeler = pd.DataFrame(denemeler).sort_values("MAE")
en_iyi = denemeler.iloc[0][list(izgara)].to_dict()
en_iyi = {k: (int(v) if k != "learning_rate" else float(v)) for k, v in en_iyi.items()}
print("En iyi ayar:", en_iyi)
denemeler.head(5).round(3)

## Son modeller: 2011-2022 ile eğit, 2023'te test et

In [ ]:
mevsim = p["egitim_tum"].groupby(p["egitim_tum"].index.dayofyear)["hedef"].mean()
modeller = {
    "Doğrusal Regresyon": LinearRegression(),
    "XGBoost (varsayılan)": XGBRegressor(random_state=0),
    "XGBoost (ayarlı)": XGBRegressor(**en_iyi, subsample=0.8, colsample_bytree=0.8, random_state=0),
}
sonuc = {
    "Taban: değişmez": olc(p["test"], np.zeros(len(p["test"]))),
    "Taban: mevsimsel": olc(p["test"], p["test"].index.dayofyear.map(mevsim).fillna(0).values),
}
tahminler = {}
for ad, m in modeller.items():
    m.fit(p["egitim_tum"][oz], p["egitim_tum"]["hedef"])
    tahminler[ad] = m.predict(p["test"][oz])
    sonuc[ad] = olc(p["test"], tahminler[ad])
tablo = pd.DataFrame(sonuc).T.round(2)
tablo

In [ ]:
gercek = p["test"]["doluluk"] + p["test"]["hedef"]
plt.figure(figsize=(12, 4))
plt.plot(gercek.index, gercek, label="gerçek", color="black")
for ad in ["Doğrusal Regresyon", "XGBoost (ayarlı)"]:
    plt.plot(gercek.index, p["test"]["doluluk"] + tahminler[ad], label=ad)
plt.title(f"2023 testi: {UFUK} gün sonraki toplam doluluk (%)"); plt.ylabel("%"); plt.legend(); plt.show()

xgb = modeller["XGBoost (ayarlı)"]
pd.Series(xgb.feature_importances_, index=oz).sort_values().plot.barh(figsize=(6, 5), title="XGBoost öznitelik önemi")
plt.show()

## Notlar (rapor için)
- Tüm modeller "değişmez" tabanıyla kıyaslanmalı; tabanı geçemeyen model işe yaramıyor demektir.
- 2023 alışılmadık kurak bir yıldı; ağaç modelleri eğitimde görmedikleri düşük doluluk seviyelerinde zorlanabilir
  (ağaçlar eğitim aralığının dışına çıkamaz, doğrusal model çıkabilir).